<a href="https://colab.research.google.com/github/nickkhymenko/asteria-credit-risk-ml/blob/main/04_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04: Modelling and Evaluation

**Project:** Asteria Finance, Repayment Difficulty Prediction (IB99Z0)

**CRISP-DM phases:** Modelling and Evaluation

**Input:** `features_train.parquet` (and `features_test.parquet` if available) from notebook 03.

## Approach: champion vs challenger

Lenders do not pick a model on a single leaderboard number. They compare an interpretable **champion** against more flexible **challengers** and only switch when the gain justifies the loss of transparency. We follow the same practice with three classical models:

| Model | Why it is in the comparison |
|---|---|
| **Scorecard**: weight-of-evidence (WoE) binning + logistic regression | The industry standard for retail credit. Every point of the score can be explained to a customer or a regulator. |
| **Random forest** | Bagged trees, as covered in the module. Captures non-linearities and interactions with little tuning. |
| **LightGBM with monotonic constraints** | Gradient boosted trees are the strongest classical method on tabular data (Grinsztajn et al., 2022). Monotonic constraints force economically sensible behaviour, for example risk can never *rise* as the external credit score improves. |

## Evaluation principles

* **Accuracy is not used.** Only $8.07\%$ of applicants have repayment difficulty, so predicting "no difficulty" for everyone already scores $91.9\%$.
* **Ranking quality:** ROC AUC and its credit-industry form, the **Gini coefficient** $= 2 \times \text{AUC} - 1$, plus the **KS statistic**.
* **Minority-class focus:** PR AUC (average precision). A random model scores about $0.08$ here, not $0.5$.
* **Probability quality:** Brier score and a calibration plot, because the predicted probability of difficulty (PD) drives pricing and provisioning.
* **Business value:** the decision threshold is chosen from an **expected profit** curve, not a default of $0.5$.
* **Robustness:** 5-fold stratified cross-validation on the training set, bootstrap confidence intervals, and a held-out test set used **once** at the end.
* **No resampling (SMOTE etc.).** It distorts predicted probabilities. Imbalance is handled through the metric and the threshold instead.

In [ ]:
# %pip install -q lightgbm shap scikit-learn pyarrow   # uncomment in a fresh environment
import os
import sys
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", "{:.4f}".format)

IN_COLAB = "google.colab" in sys.modules
DATA_DIR = Path(os.environ.get("ASTERIA_DATA_DIR", "/content" if IN_COLAB else "data"))
FEATURE_DIR = DATA_DIR / "processed"
PREDICTIONS_PATH = Path(os.environ.get("ASTERIA_PREDICTIONS_PATH", "predictions.csv"))

SEED = 42
N_FOLDS = 5
TUNE_LIGHTGBM = True      # set False for a quick run
N_BOOTSTRAP = 500

# Business assumptions for the profit curve. These are ASSUMPTIONS: state them in the report
# and show the board how the recommended cut-off moves if they change.
LGD = 0.45            # loss given default: Basel II foundation IRB value for senior unsecured exposures
MARGIN = 0.08         # net lifetime margin earned on a loan that is repaid, as a share of credit

train_df = pd.read_parquet(FEATURE_DIR / "features_train.parquet")
test_path = FEATURE_DIR / "features_test.parquet"
score_df = pd.read_parquet(test_path) if test_path.exists() else None

print("Labelled applicants:", train_df.shape)
print("Applicants to score:", None if score_df is None else score_df.shape)
print(f"Repayment difficulty rate: {train_df['TARGET'].mean():.4f}")

## 1. Prepare features and split

* `SK_ID_CURR` is an identifier, not a predictor, so it is removed.
* Text columns become pandas `category` so LightGBM can use them natively; the other models encode them inside their own pipelines.
* **$20\%$ of applicants are held out** (stratified on `TARGET`) and not touched until the final evaluation. All model selection uses cross-validation on the other $80\%$.

In [ ]:
ID, TARGET = "SK_ID_CURR", "TARGET"

X_all = train_df.drop(columns=[ID, TARGET])
y_all = train_df[TARGET].astype(int)

CAT_COLS = [c for c in X_all.columns if not is_numeric_dtype(X_all[c])]
NUM_COLS = [c for c in X_all.columns if c not in CAT_COLS]

CATEGORY_LEVELS = {c: sorted(X_all[c].dropna().astype(str).unique()) for c in CAT_COLS}


def to_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    # Apply the same column order and category levels to any applicant table.
    out = df.reindex(columns=X_all.columns).copy()
    for c in CAT_COLS:
        out[c] = pd.Categorical(out[c].astype("string"), categories=CATEGORY_LEVELS[c])
    for c in NUM_COLS:
        out[c] = out[c].astype(float)
    return out


X_all = to_model_frame(X_all)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED
)
ids_test = train_df.loc[X_test.index, ID]

print(f"{len(NUM_COLS)} numeric and {len(CAT_COLS)} categorical features")
print(f"Train: {X_train.shape}, difficulty rate {y_train.mean():.4f}")
print(f"Test:  {X_test.shape}, difficulty rate {y_test.mean():.4f}")

## 2. Model definitions

### 2.1 Scorecard: weight-of-evidence encoding + logistic regression

For each feature, values are grouped into bins (deciles for numeric features, categories for text, and a separate bin for missing values). Each bin is replaced by its **weight of evidence**:

$$\text{WoE}_{\text{bin}} = \ln\left(\frac{\text{share of non-difficulty applicants in bin}}{\text{share of difficulty applicants in bin}}\right)$$

and each feature's **information value** (IV) measures its total predictive strength:

$$\text{IV} = \sum_{\text{bins}} \left(\text{share}_{\text{good}} - \text{share}_{\text{bad}}\right) \times \text{WoE}$$

Why this suits credit risk:
* it handles missing values, outliers and non-linear relationships without imputation or scaling;
* features with $\text{IV} < 0.02$ (no predictive power) are dropped, highly correlated WoE features are pruned, and at most $15$ features are kept, giving a short, stable model like the scorecards lenders use;
* the result converts directly into a **points-based scorecard**.

The encoder is fitted inside the pipeline, so in cross-validation it only ever learns from the training folds.

In [ ]:
class WoEEncoder(BaseEstimator, TransformerMixin):
    # Weight-of-evidence binning with information-value selection and correlation pruning.

    def __init__(self, n_bins=10, smoothing=0.5, min_iv=0.02, max_corr=0.7, max_features=15,
                 rare_share=0.01):
        self.n_bins = n_bins
        self.smoothing = smoothing
        self.min_iv = min_iv
        self.max_corr = max_corr
        self.max_features = max_features
        self.rare_share = rare_share

    def _bin(self, s, spec):
        if spec["kind"] == "numeric":
            codes = pd.cut(s.astype(float), spec["edges"], labels=False, include_lowest=True)
            return codes.map(lambda v: "MISSING" if pd.isna(v) else f"B{int(v):02d}")
        labels = s.astype("string").fillna("MISSING").astype(str)
        return labels.where(labels.isin(spec["keep"]), "RARE")

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.asarray(y)
        n_bad, n_good = y.sum(), len(y) - y.sum()
        self.specs_, self.woe_, self.iv_, self.bins_ = {}, {}, {}, {}
        for col in X.columns:
            s = X[col]
            if is_numeric_dtype(s) and s.nunique() > self.n_bins:
                q = np.nanquantile(s.astype(float), np.linspace(0, 1, self.n_bins + 1))
                edges = np.unique(q)
                edges[0], edges[-1] = -np.inf, np.inf
                spec = {"kind": "numeric", "edges": edges}
            else:
                labels = s.astype("string").fillna("MISSING").astype(str)
                share = labels.value_counts(normalize=True)
                spec = {"kind": "categorical", "keep": set(share[share >= self.rare_share].index)}
            binned = self._bin(s, spec)
            tab = pd.crosstab(binned, y).reindex(columns=[0, 1], fill_value=0)
            good = (tab[0] + self.smoothing) / (n_good + self.smoothing * len(tab))
            bad = (tab[1] + self.smoothing) / (n_bad + self.smoothing * len(tab))
            woe = np.log(good / bad)
            self.specs_[col] = spec
            self.woe_[col] = woe.to_dict()
            self.iv_[col] = float(((good - bad) * woe).sum())
            self.bins_[col] = pd.DataFrame({"n": tab.sum(axis=1), "bad_rate": tab[1] / tab.sum(axis=1),
                                            "woe": woe})

        # Keep informative features, then drop the weaker one of any highly correlated pair.
        ranked = sorted((c for c in X.columns if self.iv_[c] >= self.min_iv),
                        key=lambda c: self.iv_[c], reverse=True)
        encoded = self._encode(X[ranked]) if ranked else pd.DataFrame(index=X.index)
        corr = encoded.corr().abs()
        selected = []
        for c in ranked:
            if len(selected) == self.max_features:
                break
            if all(corr.loc[c, s] < self.max_corr for s in selected):
                selected.append(c)
        self.selected_ = selected
        return self

    def _encode(self, X):
        return pd.DataFrame({c: self._bin(X[c], self.specs_[c]).map(self.woe_[c]).fillna(0.0).astype(float)
                             for c in X.columns}, index=X.index)

    def transform(self, X):
        return self._encode(pd.DataFrame(X)[self.selected_]).to_numpy()

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_)


def make_scorecard():
    return Pipeline([
        ("woe", WoEEncoder()),
        ("logit", LogisticRegression(C=0.1, max_iter=2000)),
    ])

### 2.2 Random forest

Scikit-learn's random forest accepts missing values directly, so we only need to turn categories into integer codes. `min_samples_leaf` stops trees from memorising individual applicants. We deliberately do **not** use `class_weight="balanced"`: re-weighting inflates the predicted probabilities, and this model's output must be usable as a probability of difficulty for the cut-off and profit analysis.

In [ ]:
def make_random_forest():
    encode = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                                encoded_missing_value=-1), CAT_COLS)],
        remainder="passthrough",
    )
    forest = RandomForestClassifier(
        n_estimators=400, min_samples_leaf=40, max_features="sqrt",
        n_jobs=-1, random_state=SEED,
    )
    return Pipeline([("encode", encode), ("forest", forest)])

### 2.3 LightGBM with monotonic constraints

`-1` forces predicted risk to **fall** (or stay flat) as the feature rises; `+1` forces it to **rise**. We only constrain features where the direction is economically obvious and confirmed by the EDA. **Age is deliberately not constrained**: its role is examined in the fairness section instead.

In [ ]:
MONOTONE_DIRECTION = {
    # better external credit scores -> lower risk
    "EXT_SOURCE_1": -1, "EXT_SOURCE_2": -1, "EXT_SOURCE_3": -1,
    "EXT_SOURCE_MEAN": -1, "EXT_SOURCE_MIN": -1, "EXT_SOURCE_MAX": -1,
    # worse past behaviour -> higher risk
    "PREV_REFUSAL_RATE": 1, "PREV_REFUSED_COUNT": 1,
    "INS_LATE_RATE": 1, "INS_LATE_30D_RATE": 1, "INS_DAYS_LATE_MEAN": 1,
    "BUR_OVERDUE_COUNT": 1, "BUR_MAX_DAYS_OVERDUE": 1,
    "BB_DLQ_RATE": 1, "BB_WORST_STATUS": 1,
    "POS_DPD_RATE": 1,
}
MONOTONE = [MONOTONE_DIRECTION.get(c, 0) for c in X_all.columns]

LGB_BASE = dict(
    objective="binary", n_estimators=600, learning_rate=0.03, num_leaves=31,
    min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_lambda=1.0, monotone_constraints=MONOTONE, monotone_constraints_method="advanced",
    random_state=SEED, n_jobs=-1, verbose=-1,
)


def make_lightgbm(**params):
    return lgb.LGBMClassifier(**{**LGB_BASE, **params})


print("Constrained features present:", sum(m != 0 for m in MONOTONE))

### 2.4 Hyperparameter tuning (LightGBM)

A randomised search with 3-fold CV on the training set, optimising **average precision** because the business cares most about correctly ranking the minority class. It is deliberately small: the brief rewards sound method over squeezing out the last $0.001$ of AUC.

In [ ]:
LGB_PARAMS = {}
if TUNE_LIGHTGBM:
    search = RandomizedSearchCV(
        make_lightgbm(),
        param_distributions={
            "num_leaves": [15, 31, 63],
            "min_child_samples": [20, 50, 100, 200],
            "learning_rate": [0.02, 0.03, 0.05],
            "n_estimators": [300, 600, 900],
            "colsample_bytree": [0.5, 0.7, 0.9],
            "reg_lambda": [0.0, 1.0, 5.0, 10.0],
        },
        n_iter=12, scoring="average_precision",
        cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
        random_state=SEED, n_jobs=1,
    )
    search.fit(X_train, y_train)
    LGB_PARAMS = search.best_params_
    print("Best params:", LGB_PARAMS)
    print(f"Best CV average precision: {search.best_score_:.4f}")

MODELS = {
    "Scorecard (WoE + logistic)": make_scorecard,
    "Random forest": make_random_forest,
    "LightGBM (monotone)": lambda: make_lightgbm(**LGB_PARAMS),
}

## 3. Cross-validated comparison

Each model is trained 5 times, each time predicting the fold it did not see. This gives an **out-of-fold** prediction for every training applicant, from which we compute all metrics and bootstrap $95\%$ confidence intervals.

In [ ]:
def ks_statistic(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))


def evaluate(y, p):
    auc = roc_auc_score(y, p)
    return {
        "ROC AUC": auc,
        "Gini": 2 * auc - 1,
        "PR AUC": average_precision_score(y, p),
        "KS": ks_statistic(y, p),
        "Brier": brier_score_loss(y, p),
    }


def bootstrap_ci(y, p, metric=roc_auc_score, n=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    stats = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        stats.append(metric(y[idx], p[idx]))
    return np.percentile(stats, [2.5, 97.5])


cv = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof = {name: np.zeros(len(y_train)) for name in MODELS}
fold_auc = {name: [] for name in MODELS}

for fold, (tr, va) in enumerate(cv.split(X_train, y_train), start=1):
    for name, make in MODELS.items():
        model = make().fit(X_train.iloc[tr], y_train.iloc[tr])
        p = model.predict_proba(X_train.iloc[va])[:, 1]
        oof[name][va] = p
        fold_auc[name].append(roc_auc_score(y_train.iloc[va], p))
    print(f"fold {fold} done")

rows = []
for name in MODELS:
    lo, hi = bootstrap_ci(y_train, oof[name])
    rows.append({"Model": name, **evaluate(y_train, oof[name]),
                 "AUC fold std": np.std(fold_auc[name]), "AUC 95% CI": f"[{lo:.3f}, {hi:.3f}]"})
cv_results = pd.DataFrame(rows).set_index("Model")
cv_results

### Is the best challenger really better than the scorecard?

A paired bootstrap of the AUC difference on the same applicants. If the interval contains $0$, the gain is not statistically reliable and the simpler, more transparent scorecard should be preferred.

In [ ]:
SCORECARD = "Scorecard (WoE + logistic)"
best_challenger = cv_results.drop(index=SCORECARD)["ROC AUC"].idxmax()

rng = np.random.default_rng(SEED)
y_arr = y_train.to_numpy()
diffs = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    diffs.append(roc_auc_score(y_arr[idx], oof[best_challenger][idx])
                 - roc_auc_score(y_arr[idx], oof[SCORECARD][idx]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"{best_challenger} minus scorecard, AUC difference: {np.mean(diffs):+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")

CHAMPION = best_challenger if lo > 0 else SCORECARD
print("\nSelected champion (decided on cross-validation only):", CHAMPION)

## 4. Final evaluation on the held-out test set

All three models are refitted on the full training set and scored **once** on the $20\%$ hold-out. These are the numbers to report to the board.

In [ ]:
final_models = {name: make().fit(X_train, y_train) for name, make in MODELS.items()}
test_pred = {name: m.predict_proba(X_test)[:, 1] for name, m in final_models.items()}

holdout = pd.DataFrame({name: evaluate(y_test, p) for name, p in test_pred.items()}).T
holdout["AUC 95% CI"] = [
    "[{:.3f}, {:.3f}]".format(*bootstrap_ci(y_test, test_pred[n])) for n in holdout.index
]
holdout

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, p in test_pred.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, label=f"{name} (AP {average_precision_score(y_test, p):.3f})")
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac_pos, marker="o", label=name)

axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(title="ROC curve (held-out)", xlabel="False positive rate", ylabel="True positive rate")
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label="No-skill baseline")
axes[1].set(title="Precision-recall curve (held-out)", xlabel="Recall", ylabel="Precision")
lim = max(max(p.max() for p in test_pred.values()), 0.3)
axes[2].plot([0, lim], [0, lim], "k--", lw=1, label="Perfect calibration")
axes[2].set(title="Calibration (held-out)", xlabel="Predicted probability", ylabel="Observed rate")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Reading the calibration plot:** points on the diagonal mean that, for example, applicants given a $20\%$ probability really do have difficulty about $20\%$ of the time. All three models are trained on the true class balance (no re-weighting or resampling), so their probabilities can be used directly as PDs. If a model sits well off the diagonal, recalibrate it (for example with `CalibratedClassifierCV`) before using it for pricing.

## 5. From scores to a lending decision

A model only creates value through the decision it drives. For each possible cut-off we approve applicants whose predicted probability is below it, and estimate profit on the hold-out set:

$$\text{Profit} = \sum_{\text{approved, repaid}} \text{MARGIN} \times \text{credit} \;-\; \sum_{\text{approved, difficulty}} \text{LGD} \times \text{credit}$$

with exposure at default taken as `AMT_CREDIT`. The **cut-off table** is the format credit committees actually use.

In [ ]:
p_champ = test_pred[CHAMPION]
credit = X_test["AMT_CREDIT"].to_numpy()
y_np = y_test.to_numpy()


def portfolio_outcome(threshold):
    approved = p_champ <= threshold
    good = approved & (y_np == 0)
    bad = approved & (y_np == 1)
    return {
        "approval rate": approved.mean(),
        "bad rate (approved)": y_np[approved].mean() if approved.any() else np.nan,
        "bads caught (recall)": 1 - bad.sum() / max((y_np == 1).sum(), 1),
        "profit": MARGIN * credit[good].sum() - LGD * credit[bad].sum(),
    }


cutoff_table = pd.DataFrame(
    {f"{int(q * 100)}%": portfolio_outcome(np.quantile(p_champ, q))
     for q in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]}
).T
cutoff_table.index.name = "approve lowest-risk"
baseline_profit = portfolio_outcome(1.0)["profit"]
cutoff_table["profit vs approve-all"] = cutoff_table["profit"] - baseline_profit
cutoff_table.style.format({"approval rate": "{:.1%}", "bad rate (approved)": "{:.2%}",
                           "bads caught (recall)": "{:.1%}", "profit": "{:,.0f}",
                           "profit vs approve-all": "{:+,.0f}"})

In [ ]:
grid = np.unique(np.quantile(p_champ, np.linspace(0.3, 1.0, 141)))
profits = np.array([portfolio_outcome(t)["profit"] for t in grid])
BEST_THRESHOLD = float(grid[profits.argmax()])
best = portfolio_outcome(BEST_THRESHOLD)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([portfolio_outcome(t)["approval rate"] for t in grid], profits / 1e6)
ax.axvline(best["approval rate"], color="k", ls="--", lw=1)
ax.set(title=f"Expected profit by approval rate ({CHAMPION}, held-out)",
       xlabel="Approval rate", ylabel="Profit (millions)")
plt.tight_layout()
plt.show()

print(f"Profit-maximising cut-off: PD <= {BEST_THRESHOLD:.3f}")
print(f"Approval rate {best['approval rate']:.1%}, bad rate among approved {best['bad rate (approved)']:.2%} "
      f"(vs {y_np.mean():.2%} if everyone is approved)")
print(f"Uplift vs approving everyone: {best['profit'] - baseline_profit:+,.0f}")

tn, fp, fn, tp = confusion_matrix(y_np, (p_champ > BEST_THRESHOLD).astype(int)).ravel()
print(f"\nAt this cut-off: declined {tp + fp:,} applicants, of whom {tp:,} would have had difficulty "
      f"(precision {tp / max(tp + fp, 1):.1%}); {fn:,} difficulty cases still approved.")

**Sensitivity:** `MARGIN` and `LGD` are assumptions. Re-run with different values (for example `MARGIN = 0.05` or `LGD = 0.6`) to show the board how the recommended approval rate moves. A lower margin or a higher loss rate pushes the optimal cut-off towards approving fewer applicants.

## 6. Explaining the model

### 6.1 Scorecard: information value and points

The scorecard is converted to points using the standard industry scaling: $600$ points at odds of $50{:}1$ (repaid : difficulty), with every $20$ points doubling the odds (*points to double the odds*, PDO).

In [ ]:
woe_step = final_models[SCORECARD].named_steps["woe"]
logit = final_models[SCORECARD].named_steps["logit"]

iv = pd.Series(woe_step.iv_).sort_values(ascending=False)
iv_table = pd.DataFrame({"IV": iv, "selected": iv.index.isin(woe_step.selected_)})
iv_table["strength"] = pd.cut(iv_table["IV"], [-np.inf, 0.02, 0.1, 0.3, 0.5, np.inf],
                              labels=["useless", "weak", "medium", "strong", "suspicious (check leakage)"])
print(f"{len(woe_step.selected_)} of {len(iv)} features kept in the scorecard")
iv_table.head(20)

In [ ]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
n_feat = len(woe_step.selected_)
coefs = dict(zip(woe_step.selected_, logit.coef_[0]))

points_rows = []
for feat in woe_step.selected_[:6]:
    for bin_label, row in woe_step.bins_[feat].iterrows():
        pts = -(coefs[feat] * row["woe"] + logit.intercept_[0] / n_feat) * factor + offset / n_feat
        # logistic models P(difficulty), so a higher score = lower risk after the sign flip above
        points_rows.append({"feature": feat, "bin": bin_label, "applicants": int(row["n"]),
                            "difficulty rate": row["bad_rate"], "points": round(pts)})
scorecard_points = pd.DataFrame(points_rows)
scorecard_points.head(40)

### 6.2 LightGBM: SHAP values

SHAP attributes each prediction to the features that drove it. Globally this shows what the model relies on; for a single applicant it produces **reason codes**, which a lender needs in order to explain a decline.

In [ ]:
import shap

lgbm = final_models["LightGBM (monotone)"]
sample = X_test.sample(min(2000, len(X_test)), random_state=SEED)
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(sample)
if isinstance(shap_values, list):  # older shap versions return one array per class
    shap_values = shap_values[1]

shap.summary_plot(shap_values, sample, max_display=15, show=False)
plt.title("LightGBM: top drivers of predicted repayment difficulty")
plt.tight_layout()
plt.show()

In [ ]:
# Reason codes for the three highest-risk applicants in the sample
risk = lgbm.predict_proba(sample)[:, 1]
for i in np.argsort(risk)[::-1][:3]:
    contrib = pd.Series(shap_values[i], index=sample.columns).sort_values(ascending=False)
    reasons = ", ".join(f"{f} = {sample.iloc[i][f]:.3g}" if isinstance(sample.iloc[i][f], (int, float))
                        else f"{f} = {sample.iloc[i][f]}" for f in contrib.index[:3])
    print(f"Applicant {train_df.loc[sample.index[i], ID]}: PD {risk[i]:.1%}. Main reasons: {reasons}")

## 7. Fairness and financial inclusion

Age and sex are **protected characteristics** under the UK Equality Act 2010, and the FCA Consumer Duty requires firms to avoid foreseeable harm to groups of customers. We check:

1. **Outcomes by group** at the recommended cut-off (approval rates and error rates).
2. **Ranking quality by group** (does the model work equally well for everyone?).
3. **The cost of removing protected attributes:** retrain LightGBM without age and gender and compare.
4. **Thin-file applicants:** people with no bureau record are both riskier and the group most at risk of financial exclusion.

In [ ]:
audit = pd.DataFrame({
    "y": y_np,
    "p": p_champ,
    "declined": p_champ > BEST_THRESHOLD,
    "age group": pd.cut(X_test["AGE_YEARS"], [18, 25, 35, 45, 55, 65, 100]).astype(str),
    "gender": X_test["CODE_GENDER"].astype(str) if "CODE_GENDER" in X_test else "n/a",
    "credit history": np.where(X_test["HAS_BUREAU_RECORDS"] == 1, "bureau record", "thin file"),
}, index=X_test.index)


def group_report(col):
    def stats(g):
        good, bad = g[g["y"] == 0], g[g["y"] == 1]
        return pd.Series({
            "applicants": len(g),
            "actual difficulty rate": g["y"].mean(),
            "approval rate": 1 - g["declined"].mean(),
            "good customers wrongly declined": good["declined"].mean() if len(good) else np.nan,
            "difficulty cases missed": (~bad["declined"]).mean() if len(bad) else np.nan,
            "AUC within group": roc_auc_score(g["y"], g["p"]) if g["y"].nunique() == 2 else np.nan,
        })
    return audit.groupby(col).apply(stats)


for col in ["age group", "gender", "credit history"]:
    display(group_report(col).style.format("{:.3f}").format({"applicants": "{:,.0f}"}))

In [ ]:
PROTECTED = [c for c in ["DAYS_BIRTH", "AGE_YEARS", "EMPLOYED_TO_AGE_RATIO", "CODE_GENDER"] if c in X_train]
X_train_fair = X_train.drop(columns=PROTECTED)
X_test_fair = X_test.drop(columns=PROTECTED)
mono_fair = [MONOTONE_DIRECTION.get(c, 0) for c in X_train_fair.columns]

fair_model = make_lightgbm(**LGB_PARAMS).set_params(monotone_constraints=mono_fair)
fair_model.fit(X_train_fair, y_train)
p_fair = fair_model.predict_proba(X_test_fair)[:, 1]

comparison = pd.DataFrame({
    "with age and gender": evaluate(y_test, test_pred["LightGBM (monotone)"]),
    "without age and gender": evaluate(y_test, p_fair),
}).T
print("Removed:", PROTECTED)
comparison

**How to discuss this in the report:** removing protected attributes rarely removes their influence entirely, because other features (years employed, family status) act as proxies. The group tables above are therefore the real test. If the AUC cost of dropping age and gender is small, recommending the version without them is a defensible, low-regret choice for the board.

## 8. Predictions for the unlabelled applicants

The champion is refitted on **all** labelled data (training + hold-out) before scoring, so the final model learns from every available example. The column `TARGET` holds the predicted probability of repayment difficulty.

> **Check the brief:** confirm whether `predictions.csv` should contain probabilities or 0/1 labels. For labels, use the profit-maximising cut-off from section 5.

In [ ]:
if score_df is None:
    print("No features_test.parquet found: set TEST_FILE in notebook 03 once the test file is confirmed.")
else:
    final_champion = MODELS[CHAMPION]().fit(X_all, y_all)
    X_score = to_model_frame(score_df.drop(columns=[c for c in [ID, TARGET] if c in score_df]))
    proba = final_champion.predict_proba(X_score)[:, 1]
    predictions = pd.DataFrame({ID: score_df[ID].to_numpy(), TARGET: proba})
    predictions["DECISION"] = np.where(proba > BEST_THRESHOLD, "decline", "approve")
    predictions.to_csv(PREDICTIONS_PATH, index=False)
    print(f"Saved {len(predictions):,} predictions to {PREDICTIONS_PATH.resolve()}")
    print(predictions[TARGET].describe())

## 9. Summary for the report and presentation

* **Champion selection** was made on cross-validation only, using a paired bootstrap test against the scorecard. The hold-out set was used once, for the final figures.
* **Headline metrics** to quote: hold-out ROC AUC / Gini with its $95\%$ confidence interval, PR AUC against the $0.08$ baseline, and the calibration plot.
* **Business recommendation:** the cut-off table and profit curve translate the model into an approval policy, together with the stated `MARGIN` and `LGD` assumptions and their sensitivity.
* **Transparency:** scorecard points and SHAP reason codes show why an applicant is declined.
* **Responsible lending:** group-level outcomes for age, gender and thin-file applicants, and the cost of removing protected attributes.

**Limitations to state:**
* Only *approved* past loans have observed outcomes, so the model never learns from people who were declined (in industry this is called *reject inference*).
* There is no application date, so an out-of-time validation (train on older applications, test on newer) was not possible. Performance should be monitored after deployment, for example with the population stability index (PSI).
* The profit model uses simplified, assumed `MARGIN` and `LGD` values.